# Kaggle Submit - Standalone XGBoost

이 노트북 하나만으로 전처리, 학습, 제출까지 진행합니다.

- 외부 helper import 없음
- `iter_test()` 제출
- `env.predict(...)` 사용


In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error

try:
    import torch
    GPU_AVAILABLE = torch.cuda.is_available()
except Exception:
    GPU_AVAILABLE = False

DEBUG = False
DATA_DIR = '/kaggle/input/predict-energy-behavior-of-prosumers'
TARGET_COLUMN = 'target'
GROUP_COLS = ['prediction_unit_id', 'is_consumption']
STATIC_GROUP_COLS = ['county', 'is_business', 'product_type']

CLIENT_COLUMNS = ['county', 'is_business', 'product_type', 'date', 'eic_count', 'installed_capacity']
HISTORICAL_WEATHER_COLUMNS = [
    'datetime', 'temperature', 'dewpoint', 'rain', 'snowfall', 'surface_pressure',
    'cloudcover_total', 'windspeed_10m', 'shortwave_radiation', 'direct_solar_radiation',
    'latitude', 'longitude'
]
WEATHER_BASE_COLUMNS = [
    'temperature', 'dewpoint', 'rain', 'snowfall', 'surface_pressure',
    'cloudcover_total', 'windspeed_10m', 'shortwave_radiation', 'direct_solar_radiation'
]
COUNTY_WEATHER_COLUMNS = ['temp_county', 'solar_county', 'direct_solar', 'cloud_county', 'wind_county']
INTERP_COLS = [
    'temperature', 'dewpoint', 'rain', 'snowfall', 'surface_pressure', 'cloudcover_total',
    'windspeed_10m', 'shortwave_radiation', 'euros_per_mwh', 'lowest_price_per_mwh', 'highest_price_per_mwh'
]
LAG_IMPUTE_COLS = ['lag_1', 'lag_24']
MEAN_COLS = ['eic_count', 'installed_capacity']

BASELINE_FEATURES = [
    'county', 'is_business', 'product_type', 'hour', 'weekday', 'month', 'day', 'dayofyear', 'weekofyear',
    'temperature', 'dewpoint', 'rain', 'snowfall', 'surface_pressure', 'cloudcover_total', 'windspeed_10m',
    'shortwave_radiation', 'euros_per_mwh', 'lowest_price_per_mwh', 'highest_price_per_mwh',
    'temp_county', 'solar_county', 'direct_solar', 'cloud_county', 'wind_county', 'lag_1', 'lag_24'
]
ENGINEERED_FEATURES = [
    'quarter', 'is_weekend', 'is_month_start', 'is_month_end', 'is_holiday', 'day_before_holiday',
    'day_after_holiday', 'n_holidays_last_7d', 'n_holidays_last_14d', 'is_quarter_start',
    'is_quarter_end', 'days_in_month', 'lag_2', 'lag_3', 'lag_6', 'lag_12', 'lag_48', 'lag_72',
    'lag_96', 'lag_120', 'lag_168', 'target_diff_1_2', 'target_diff_1_24', 'target_diff_24_48',
    'target_diff_24_168', 'target_diff_1_168', 'target_ratio_1_2', 'target_ratio_1_24',
    'target_ratio_24_48', 'target_ratio_24_168', 'target_ratio_1_168', 'target_trend_1',
    'target_trend_24', 'same_hour_mean_3d', 'same_hour_std_3d', 'same_hour_mean_7d',
    'same_hour_std_7d', 'same_hour_mean_14d', 'same_hour_std_14d', 'same_hour_ratio_7d',
    'temp_diff_1', 'temp_diff_24', 'temp_diff_168', 'solar_diff_24', 'solar_diff_168',
    'cloud_diff_24', 'wind_diff_24', 'rolling_mean_168_temp', 'rolling_mean_168_solar',
    'temp_vs_rolling_168', 'solar_vs_rolling_168', 'installed_capacity_x_shortwave_radiation',
    'installed_capacity_x_direct_solar_radiation', 'is_business_x_hour', 'is_consumption_x_temperature',
    'is_consumption_x_hour', 'temperature_x_hour', 'cloudcover_x_solar', 'price_diff', 'price_ratio',
    'rolling_mean_3', 'rolling_std_3', 'rolling_mean_6', 'rolling_std_6', 'rolling_mean_12',
    'rolling_std_12', 'rolling_mean_24', 'rolling_std_24', 'rolling_mean_48', 'rolling_std_48',
    'rolling_mean_168', 'rolling_std_168', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'month_sin', 'month_cos', 'day_sin', 'day_cos', 'dayofyear_sin', 'dayofyear_cos',
    'weekofyear_sin', 'weekofyear_cos', 'quarter_sin', 'quarter_cos'
]
FEATURE_COLUMNS = BASELINE_FEATURES + ENGINEERED_FEATURES

XGB_PARAMS = dict(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    tree_method='hist',
    device='cuda' if GPU_AVAILABLE else 'cpu',
    random_state=42,
    early_stopping_rounds=50,
    eval_metric='mae',
)

print('config ready')
print('GPU available:', GPU_AVAILABLE)


In [ ]:
def load_competition_tables(data_dir=DATA_DIR):
    return {
        'train': pd.read_csv(f'{data_dir}/train.csv'),
        'client': pd.read_csv(f'{data_dir}/client.csv'),
        'historical_weather': pd.read_csv(f'{data_dir}/historical_weather.csv'),
        'forecast_weather': pd.read_csv(f'{data_dir}/forecast_weather.csv'),
        'electricity': pd.read_csv(f'{data_dir}/electricity_prices.csv'),
        'gas': pd.read_csv(f'{data_dir}/gas_prices.csv'),
        'weather_station_to_county': pd.read_csv(f'{data_dir}/weather_station_to_county_mapping.csv'),
    }

def est_public_holidays(year):
    easter = pd.Timestamp(year=year, month=3, day=21) + pd.offsets.Easter()
    fixed = [
        pd.Timestamp(year=year, month=1, day=1),
        pd.Timestamp(year=year, month=2, day=24),
        pd.Timestamp(year=year, month=5, day=1),
        pd.Timestamp(year=year, month=6, day=23),
        pd.Timestamp(year=year, month=6, day=24),
        pd.Timestamp(year=year, month=8, day=20),
        pd.Timestamp(year=year, month=12, day=24),
        pd.Timestamp(year=year, month=12, day=25),
        pd.Timestamp(year=year, month=12, day=26),
    ]
    movable = [easter - pd.Timedelta(days=2), easter, easter + pd.Timedelta(days=49)]
    return fixed + movable

def _normalize_datetime_columns(frame, datetime_col='datetime', date_col='date'):
    frame = frame.copy()
    if datetime_col in frame.columns:
        frame[datetime_col] = pd.to_datetime(frame[datetime_col])
        frame[date_col] = frame[datetime_col].dt.normalize()
    elif date_col in frame.columns:
        frame[date_col] = pd.to_datetime(frame[date_col]).dt.normalize()
    return frame

def add_time_features(frame):
    frame = _normalize_datetime_columns(frame)
    years = range(frame['datetime'].dt.year.min(), frame['datetime'].dt.year.max() + 1)
    holiday_dates = pd.DatetimeIndex(sorted({d.normalize() for year in years for d in est_public_holidays(int(year))}))
    date_series = frame['datetime'].dt.normalize()
    frame['hour'] = frame['datetime'].dt.hour.astype('int8')
    frame['weekday'] = frame['datetime'].dt.weekday.astype('int8')
    frame['month'] = frame['datetime'].dt.month.astype('int8')
    frame['day'] = frame['datetime'].dt.day.astype('int8')
    frame['dayofyear'] = frame['datetime'].dt.dayofyear.astype('int16')
    frame['weekofyear'] = frame['datetime'].dt.isocalendar().week.astype('int16')
    frame['quarter'] = frame['datetime'].dt.quarter.astype('int8')
    frame['is_weekend'] = (frame['weekday'] >= 5).astype('int8')
    frame['is_month_start'] = frame['datetime'].dt.is_month_start.astype('int8')
    frame['is_month_end'] = frame['datetime'].dt.is_month_end.astype('int8')
    frame['is_quarter_start'] = frame['datetime'].dt.is_quarter_start.astype('int8')
    frame['is_quarter_end'] = frame['datetime'].dt.is_quarter_end.astype('int8')
    frame['days_in_month'] = frame['datetime'].dt.days_in_month.astype('int8')
    frame['is_holiday'] = date_series.isin(holiday_dates).astype('int8')
    frame['day_before_holiday'] = date_series.isin(holiday_dates - pd.Timedelta(days=1)).astype('int8')
    frame['day_after_holiday'] = date_series.isin(holiday_dates + pd.Timedelta(days=1)).astype('int8')
    holiday_daily = pd.Series(1, index=holiday_dates)
    holiday_cnt_7 = holiday_daily.rolling(7, min_periods=1).sum().rename('n_holidays_last_7d')
    holiday_cnt_14 = holiday_daily.rolling(14, min_periods=1).sum().rename('n_holidays_last_14d')
    holiday_count_df = pd.concat([holiday_cnt_7, holiday_cnt_14], axis=1).reset_index().rename(columns={'index': 'date'})
    frame = frame.merge(holiday_count_df, on='date', how='left')
    frame[['n_holidays_last_7d', 'n_holidays_last_14d']] = frame[['n_holidays_last_7d', 'n_holidays_last_14d']].fillna(0).astype('int8')
    return frame

def _clean_weather_station_mapping(mapping):
    mapping = mapping.dropna(subset=['county']).copy()
    mapping['county'] = mapping['county'].astype(int)
    return mapping[['latitude', 'longitude', 'county']]

print('time helpers ready')

In [ ]:
def aggregate_historical_weather(historical_weather, weather_station_to_county):
    historical_weather = historical_weather.copy()
    historical_weather['datetime'] = pd.to_datetime(historical_weather['datetime'])
    historical_weather = historical_weather[HISTORICAL_WEATHER_COLUMNS].copy()
    county_mapping = _clean_weather_station_mapping(weather_station_to_county)
    weather_agg = historical_weather.groupby('datetime', as_index=False).agg(
        temperature=('temperature', 'mean'), dewpoint=('dewpoint', 'mean'), rain=('rain', 'mean'),
        snowfall=('snowfall', 'mean'), surface_pressure=('surface_pressure', 'mean'),
        cloudcover_total=('cloudcover_total', 'mean'), windspeed_10m=('windspeed_10m', 'mean'),
        shortwave_radiation=('shortwave_radiation', 'mean'), direct_solar_radiation=('direct_solar_radiation', 'mean')
    )
    hw_county = historical_weather.merge(county_mapping, on=['latitude', 'longitude'], how='inner')
    weather_county = hw_county.groupby(['county', 'datetime'], as_index=False).agg(
        temp_county=('temperature', 'mean'), solar_county=('shortwave_radiation', 'mean'),
        direct_solar=('direct_solar_radiation', 'mean'), cloud_county=('cloudcover_total', 'mean'),
        wind_county=('windspeed_10m', 'mean')
    )
    return weather_agg, weather_county

def aggregate_forecast_weather(forecast_weather, weather_station_to_county):
    forecast_weather = forecast_weather.copy()
    forecast_weather['forecast_datetime'] = pd.to_datetime(forecast_weather['forecast_datetime'])
    forecast_weather['windspeed_10m'] = np.sqrt(
        forecast_weather['10_metre_u_wind_component'] ** 2 + forecast_weather['10_metre_v_wind_component'] ** 2
    )
    forecast_weather['datetime'] = forecast_weather['forecast_datetime']
    forecast_weather['rain'] = forecast_weather['total_precipitation']
    forecast_weather['shortwave_radiation'] = forecast_weather['surface_solar_radiation_downwards']
    forecast_weather['surface_pressure'] = np.nan
    forecast_weather = forecast_weather[[
        'datetime', 'temperature', 'dewpoint', 'rain', 'snowfall', 'surface_pressure',
        'cloudcover_total', 'windspeed_10m', 'shortwave_radiation', 'direct_solar_radiation', 'latitude', 'longitude'
    ]].copy()
    county_mapping = _clean_weather_station_mapping(weather_station_to_county)
    weather_agg = forecast_weather.groupby('datetime', as_index=False).agg(
        temperature=('temperature', 'mean'), dewpoint=('dewpoint', 'mean'), rain=('rain', 'mean'),
        snowfall=('snowfall', 'mean'), surface_pressure=('surface_pressure', 'mean'),
        cloudcover_total=('cloudcover_total', 'mean'), windspeed_10m=('windspeed_10m', 'mean'),
        shortwave_radiation=('shortwave_radiation', 'mean'), direct_solar_radiation=('direct_solar_radiation', 'mean')
    )
    fw_county = forecast_weather.merge(county_mapping, on=['latitude', 'longitude'], how='inner')
    weather_county = fw_county.groupby(['county', 'datetime'], as_index=False).agg(
        temp_county=('temperature', 'mean'), solar_county=('shortwave_radiation', 'mean'),
        direct_solar=('direct_solar_radiation', 'mean'), cloud_county=('cloudcover_total', 'mean'),
        wind_county=('windspeed_10m', 'mean')
    )
    return weather_agg, weather_county

print('weather helpers ready')

In [ ]:
def _combine_weather_sources(frame, historical_weather, forecast_weather, weather_station_to_county):
    frame = frame.copy()
    hist_global, hist_county = aggregate_historical_weather(historical_weather, weather_station_to_county)
    frame = frame.merge(hist_global, on='datetime', how='left')
    frame = frame.merge(hist_county, on=['county', 'datetime'], how='left')
    if forecast_weather is None:
        return frame
    fcst_global, fcst_county = aggregate_forecast_weather(forecast_weather, weather_station_to_county)
    frame = frame.merge(fcst_global, on='datetime', how='left', suffixes=('', '_forecast'))
    for col in WEATHER_BASE_COLUMNS:
        forecast_col = f'{col}_forecast'
        if forecast_col in frame.columns:
            frame[col] = frame[col].combine_first(frame[forecast_col])
            frame.drop(columns=[forecast_col], inplace=True)
    frame = frame.merge(fcst_county, on=['county', 'datetime'], how='left', suffixes=('', '_forecast'))
    for col in COUNTY_WEATHER_COLUMNS:
        forecast_col = f'{col}_forecast'
        if forecast_col in frame.columns:
            frame[col] = frame[col].combine_first(frame[forecast_col])
            frame.drop(columns=[forecast_col], inplace=True)
    return frame

def merge_external_features(frame, client, historical_weather, electricity, gas, weather_station_to_county, forecast_weather=None):
    frame = _normalize_datetime_columns(frame)
    client = _normalize_datetime_columns(client, datetime_col='date', date_col='date')
    electricity = electricity.copy()
    electricity['forecast_date'] = pd.to_datetime(electricity['forecast_date'])
    electricity = electricity[['forecast_date', 'euros_per_mwh']].rename(columns={'forecast_date': 'datetime'})
    gas = gas.copy()
    gas['forecast_date'] = pd.to_datetime(gas['forecast_date'])
    gas['date'] = gas['forecast_date'].dt.normalize()
    gas = gas[['date', 'lowest_price_per_mwh', 'highest_price_per_mwh']]
    frame = frame.merge(client[CLIENT_COLUMNS], on=['county', 'is_business', 'product_type', 'date'], how='left')
    frame = _combine_weather_sources(frame, historical_weather, forecast_weather, weather_station_to_county)
    frame = frame.merge(electricity, on='datetime', how='left')
    frame = frame.merge(gas, on='date', how='left')
    return frame

def add_target_features(frame):
    frame = frame.copy().sort_values(GROUP_COLS + ['datetime']).reset_index(drop=True)
    ratio_eps = np.float32(1e-3)
    price_eps = np.float32(1e-3)
    target_group = frame.groupby(GROUP_COLS)[TARGET_COLUMN]
    frame['lag_1'] = target_group.shift(1).astype('float32')
    frame['lag_24'] = target_group.shift(24).astype('float32')
    for lag in [2, 3, 6, 12, 48, 72, 96, 120, 168]:
        frame[f'lag_{lag}'] = target_group.shift(lag).astype('float32')
    frame['target_diff_1_2'] = (frame['lag_1'] - frame['lag_2']).astype('float32')
    frame['target_diff_1_24'] = (frame['lag_1'] - frame['lag_24']).astype('float32')
    frame['target_diff_24_48'] = (frame['lag_24'] - frame['lag_48']).astype('float32')
    frame['target_diff_24_168'] = (frame['lag_24'] - frame['lag_168']).astype('float32')
    frame['target_diff_1_168'] = (frame['lag_1'] - frame['lag_168']).astype('float32')
    frame['target_ratio_1_2'] = (frame['lag_1'] / (frame['lag_2'] + ratio_eps)).astype('float32')
    frame['target_ratio_1_24'] = (frame['lag_1'] / (frame['lag_24'] + ratio_eps)).astype('float32')
    frame['target_ratio_24_48'] = (frame['lag_24'] / (frame['lag_48'] + ratio_eps)).astype('float32')
    frame['target_ratio_24_168'] = (frame['lag_24'] / (frame['lag_168'] + ratio_eps)).astype('float32')
    frame['target_ratio_1_168'] = (frame['lag_1'] / (frame['lag_168'] + ratio_eps)).astype('float32')
    frame['target_trend_1'] = (frame['lag_1'] - frame['lag_2']).astype('float32')
    frame['target_trend_24'] = (frame['lag_24'] - frame['lag_48']).astype('float32')
    for days in [3, 7, 14]:
        same_hour_lags = [target_group.shift(24 * day).rename(f'same_hour_lag_{day}d') for day in range(1, days + 1)]
        same_hour_frame = pd.concat(same_hour_lags, axis=1)
        frame[f'same_hour_mean_{days}d'] = same_hour_frame.mean(axis=1).astype('float32')
        frame[f'same_hour_std_{days}d'] = same_hour_frame.std(axis=1).astype('float32')
    frame['same_hour_ratio_7d'] = (frame['lag_24'] / (frame['same_hour_mean_7d'] + ratio_eps)).astype('float32')
    for lag in [1, 24, 168]:
        frame[f'temp_diff_{lag}'] = (frame['temperature'] - frame.groupby(GROUP_COLS)['temperature'].shift(lag)).astype('float32')
    for lag in [24, 168]:
        frame[f'solar_diff_{lag}'] = (frame['shortwave_radiation'] - frame.groupby(GROUP_COLS)['shortwave_radiation'].shift(lag)).astype('float32')
    frame['cloud_diff_24'] = (frame['cloudcover_total'] - frame.groupby(GROUP_COLS)['cloudcover_total'].shift(24)).astype('float32')
    frame['wind_diff_24'] = (frame['windspeed_10m'] - frame.groupby(GROUP_COLS)['windspeed_10m'].shift(24)).astype('float32')
    frame['rolling_mean_168_temp'] = frame.groupby(GROUP_COLS)['temperature'].transform(lambda s: s.shift(1).rolling(168, min_periods=1).mean()).astype('float32')
    frame['rolling_mean_168_solar'] = frame.groupby(GROUP_COLS)['shortwave_radiation'].transform(lambda s: s.shift(1).rolling(168, min_periods=1).mean()).astype('float32')
    frame['temp_vs_rolling_168'] = (frame['temperature'] - frame['rolling_mean_168_temp']).astype('float32')
    frame['solar_vs_rolling_168'] = (frame['shortwave_radiation'] - frame['rolling_mean_168_solar']).astype('float32')
    frame['installed_capacity_x_shortwave_radiation'] = (frame['installed_capacity'] * frame['shortwave_radiation']).astype('float32')
    frame['installed_capacity_x_direct_solar_radiation'] = (frame['installed_capacity'] * frame['direct_solar_radiation']).astype('float32')
    frame['is_business_x_hour'] = (frame['is_business'] * frame['hour']).astype('float32')
    frame['is_consumption_x_temperature'] = (frame['is_consumption'] * frame['temperature']).astype('float32')
    frame['is_consumption_x_hour'] = (frame['is_consumption'] * frame['hour']).astype('float32')
    frame['temperature_x_hour'] = (frame['temperature'] * frame['hour']).astype('float32')
    frame['cloudcover_x_solar'] = (frame['cloudcover_total'] * frame['shortwave_radiation']).astype('float32')
    frame['price_diff'] = (frame['highest_price_per_mwh'] - frame['lowest_price_per_mwh']).astype('float32')
    frame['price_ratio'] = (frame['highest_price_per_mwh'] / (frame['lowest_price_per_mwh'] + price_eps)).astype('float32')
    for window in [3, 6, 12, 24, 48, 168]:
        frame[f'rolling_mean_{window}'] = target_group.transform(lambda s: s.shift(1).rolling(window, min_periods=1).mean()).astype('float32')
        frame[f'rolling_std_{window}'] = target_group.transform(lambda s: s.shift(1).rolling(window, min_periods=1).std()).astype('float32')
    frame['hour_sin'] = np.sin(2 * np.pi * frame['hour'] / 24).astype('float32')
    frame['hour_cos'] = np.cos(2 * np.pi * frame['hour'] / 24).astype('float32')
    frame['dow_sin'] = np.sin(2 * np.pi * frame['weekday'] / 7).astype('float32')
    frame['dow_cos'] = np.cos(2 * np.pi * frame['weekday'] / 7).astype('float32')
    frame['month_sin'] = np.sin(2 * np.pi * (frame['month'] - 1) / 12).astype('float32')
    frame['month_cos'] = np.cos(2 * np.pi * (frame['month'] - 1) / 12).astype('float32')
    frame['day_sin'] = np.sin(2 * np.pi * (frame['day'] - 1) / frame['days_in_month']).astype('float32')
    frame['day_cos'] = np.cos(2 * np.pi * (frame['day'] - 1) / frame['days_in_month']).astype('float32')
    frame['dayofyear_sin'] = np.sin(2 * np.pi * (frame['dayofyear'] - 1) / 365).astype('float32')
    frame['dayofyear_cos'] = np.cos(2 * np.pi * (frame['dayofyear'] - 1) / 365).astype('float32')
    frame['weekofyear_sin'] = np.sin(2 * np.pi * (frame['weekofyear'] - 1) / 52).astype('float32')
    frame['weekofyear_cos'] = np.cos(2 * np.pi * (frame['weekofyear'] - 1) / 52).astype('float32')
    frame['quarter_sin'] = np.sin(2 * np.pi * (frame['quarter'] - 1) / 4).astype('float32')
    frame['quarter_cos'] = np.cos(2 * np.pi * (frame['quarter'] - 1) / 4).astype('float32')
    return frame

def impute_missing_values(frame):
    frame = frame.copy().sort_values(GROUP_COLS + ['datetime']).reset_index(drop=True)
    for col in INTERP_COLS:
        if col in frame.columns:
            frame[col] = frame.groupby(GROUP_COLS)[col].transform(lambda s: s.interpolate(method='linear', limit_direction='both'))
    for col in LAG_IMPUTE_COLS:
        if col in frame.columns:
            frame[col] = frame.groupby(GROUP_COLS)[col].transform(lambda s: s.interpolate(method='linear', limit_direction='both'))
            frame[col] = frame.groupby(GROUP_COLS)[col].transform(lambda s: s.fillna(s.mean()))
    for col in MEAN_COLS:
        if col in frame.columns:
            frame[col] = frame.groupby(STATIC_GROUP_COLS)[col].transform(lambda s: s.fillna(s.mean()))
    num_cols = frame.select_dtypes(include='number').columns
    frame[num_cols] = frame[num_cols].fillna(frame[num_cols].mean())
    return frame

def build_feature_frame(frame, client, historical_weather, electricity, gas, weather_station_to_county, forecast_weather=None, history_df=None):
    current = frame.copy()
    current['__is_current'] = 1
    current['__row_order'] = np.arange(len(current))
    if history_df is not None:
        history = history_df.copy()
        history['__is_current'] = 0
        history['__row_order'] = -1
        combined = pd.concat([history, current], ignore_index=True, sort=False)
    else:
        combined = current.copy()
    combined = add_time_features(combined)
    combined = merge_external_features(combined, client, historical_weather, electricity, gas, weather_station_to_county, forecast_weather)
    combined = add_target_features(combined)
    combined = impute_missing_values(combined)
    current_only = combined.loc[combined['__is_current'] == 1].copy()
    current_only = current_only.sort_values('__row_order').drop(columns=['__is_current', '__row_order'])
    return current_only.reset_index(drop=True)

print('feature helpers ready')

## 1. Train

In [ ]:
tables = load_competition_tables(DATA_DIR)
train_processed = build_feature_frame(
    frame=tables['train'],
    client=tables['client'],
    historical_weather=tables['historical_weather'],
    forecast_weather=tables['forecast_weather'],
    electricity=tables['electricity'],
    gas=tables['gas'],
    weather_station_to_county=tables['weather_station_to_county'],
)
split_date = train_processed['datetime'].quantile(0.8)
train_df = train_processed[train_processed['datetime'] < split_date].copy()
valid_df = train_processed[train_processed['datetime'] >= split_date].copy()
print(train_processed.shape, train_df.shape, valid_df.shape)

In [ ]:
def train_xgb_single(train_df, valid_df, feature_cols, params=None, label='xgboost'):
    params = {**XGB_PARAMS, **(params or {})}
    x_train = train_df[feature_cols].fillna(0)
    y_train = train_df[TARGET_COLUMN]
    x_valid = valid_df[feature_cols].fillna(0)
    y_valid = valid_df[TARGET_COLUMN]
    model = xgb.XGBRegressor(**params)
    model.fit(x_train, y_train, eval_set=[(x_valid, y_valid)], verbose=100)
    valid_pred = np.clip(model.predict(x_valid), 0, None)
    mae = mean_absolute_error(y_valid, valid_pred)
    print(f'[{label}] valid MAE: {mae:.4f}')
    return model, valid_pred, mae

tr_cons = train_df[train_df['is_consumption'] == 1].copy()
tr_prod = train_df[train_df['is_consumption'] == 0].copy()
va_cons = valid_df[valid_df['is_consumption'] == 1].copy()
va_prod = valid_df[valid_df['is_consumption'] == 0].copy()

FEAT_CONS = [c for c in FEATURE_COLUMNS if c in tr_cons.columns]
FEAT_PROD = [c for c in FEATURE_COLUMNS if c in tr_prod.columns]

model_cons, pred_cons, mae_cons = train_xgb_single(tr_cons, va_cons, FEAT_CONS, label='consumption')
model_prod, pred_prod, mae_prod = train_xgb_single(tr_prod, va_prod, FEAT_PROD, label='production')

valid_scored = pd.concat([
    va_cons.assign(pred=np.clip(pred_cons, 0, None)),
    va_prod.assign(pred=np.clip(pred_prod, 0, None)),
], axis=0).sort_values('row_id' if 'row_id' in valid_df.columns else 'datetime')

overall_mae = mean_absolute_error(valid_scored[TARGET_COLUMN], valid_scored['pred'])
print('overall valid MAE:', overall_mae)

## 2. Submit

In [ ]:
import enefit

def _extract_target_frame(revealed_targets, previous_test_df):
    if revealed_targets is None or len(revealed_targets) == 0:
        return None
    target_frame = revealed_targets.copy()
    if TARGET_COLUMN in target_frame.columns:
        return target_frame
    if previous_test_df is None:
        return None
    target_like_cols = [col for col in target_frame.columns if 'target' in col.lower()]
    if not target_like_cols:
        return None
    target_col = target_like_cols[0]
    join_keys = [col for col in ['row_id', 'prediction_unit_id', 'datetime', 'is_consumption'] if col in target_frame.columns and col in previous_test_df.columns]
    if not join_keys:
        return None
    merged = previous_test_df.merge(target_frame[join_keys + [target_col]], on=join_keys, how='inner')
    merged = merged.rename(columns={target_col: TARGET_COLUMN})
    return merged

def update_history_with_revealed_targets(history_df, revealed_targets, previous_test_df):
    revealed_frame = _extract_target_frame(revealed_targets, previous_test_df)
    if revealed_frame is None or len(revealed_frame) == 0:
        return history_df
    shared_cols = [col for col in history_df.columns if col in revealed_frame.columns]
    updated_history = pd.concat([history_df, revealed_frame[shared_cols]], ignore_index=True, sort=False)
    dedupe_keys = [col for col in ['row_id', 'prediction_unit_id', 'datetime', 'is_consumption'] if col in updated_history.columns]
    if dedupe_keys:
        updated_history = updated_history.drop_duplicates(subset=dedupe_keys, keep='last')
    return updated_history.reset_index(drop=True)

def predict_xgb_pair(test_df):
    test_pred = test_df.copy()
    cons_mask = test_pred['is_consumption'] == 1
    prod_mask = ~cons_mask
    test_pred['prediction'] = 0.0
    if cons_mask.any():
        x_cons = test_pred.loc[cons_mask, FEAT_CONS].fillna(0)
        test_pred.loc[cons_mask, 'prediction'] = np.clip(model_cons.predict(x_cons), 0, None)
    if prod_mask.any():
        x_prod = test_pred.loc[prod_mask, FEAT_PROD].fillna(0)
        test_pred.loc[prod_mask, 'prediction'] = np.clip(model_prod.predict(x_prod), 0, None)
    return test_pred

env = enefit.make_env()
iter_test = env.iter_test()

if DEBUG:
    enefit.make_env.__called__ = False
    type(env)._state = type(type(env)._state).__dict__['INIT']
    iter_test = env.iter_test()

history_df = tables['train'].copy()
weather_station_to_county = tables['weather_station_to_county'].copy()
previous_test_df = None

for batch_idx, (
    test,
    revealed_targets,
    client_test,
    historical_weather_test,
    forecast_weather_test,
    electricity_test,
    gas_test,
    sample_prediction,
) in enumerate(iter_test):
    if 'prediction_datetime' in test.columns and 'datetime' not in test.columns:
        test = test.rename(columns={'prediction_datetime': 'datetime'})

    history_df = update_history_with_revealed_targets(history_df, revealed_targets, previous_test_df)

    test_processed = build_feature_frame(
        frame=test,
        history_df=history_df,
        client=client_test,
        historical_weather=historical_weather_test,
        forecast_weather=forecast_weather_test,
        electricity=electricity_test,
        gas=gas_test,
        weather_station_to_county=weather_station_to_county,
    )

    pred_df = predict_xgb_pair(test_processed)
    sample_prediction['target'] = pred_df['prediction'].to_numpy()
    env.predict(sample_prediction)

    previous_test_df = test.copy()

    if DEBUG and batch_idx >= 1:
        print('debug stop after 2 batches')
        break
